# Stress Prediction v25 — Session-Relative Features + Multi-Model Ensemble

## Diagnosis of v24 failure

- Best submission: `a0.8, t0.33, s0.20` → `[185,76,767]` dev=0.026 → LB 0.38872
- OOF threshold t1=0.13 gave 96 class-1 on train OOF, but **148** on test → brittle, domain shift
- Root cause: class 1 vs class 2 are on a continuous arousal scale. Model can't reliably separate them from 66 samples using only absolute signal values.

## v25 fixes

### 1. Session-relative rank features (NEW — highest expected gain)
For each label, compute where that window's HR/EDA/compound_stress ranks within the same nurse's current session.
- Class 2 = near the top of the session's arousal distribution
- Class 1 = moderate rank (above resting, below peak)
- This gives the model *relative context* it currently can't infer from absolute values.

### 2. Temporal trajectory: rising vs falling (NEW)
Is the signal rising into this label moment (approaching a stress peak) or falling (recovering)?
- Compare the last 30-sec window mean vs the 60-sec-before-that window mean.
- Encode as slope direction and magnitude relative to session mean.

### 3. CatBoost + ExtraTreesClassifier ensemble (NEW)
LightGBM alone may be overconfident on the class 1/2 boundary with sparse training.
- CatBoost: built-in ordered boosting, different bias-variance profile
- ExtraTrees: extreme randomness = high variance reduction when ensembled
- Average all three model families' probabilities before calibration

### 4. Isotonic regression calibration (replaces brittle threshold)
Train isotonic regression on OOF probabilities to reshape probability outputs.
Then use argmax (no t1 threshold) — better-shaped probabilities give better decisions.
Combined with alpha=0.8 prior (proven best).

### 5. Stable calibration: alpha=0.8, t1=0.33 as primary (proven)
The proven best config is our primary submission. v25 additions are on top.

In [1]:
%pip -q install lightgbm catboost scikit-learn pandas numpy scipy


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as spstats
from scipy import signal as sps
from scipy.integrate import trapezoid

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import label_binarize

import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR    = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes')
print('  TRAIN_DATA :', TRAIN_DATA.shape)
print('  TRAIN_LABEL:', TRAIN_LABEL.shape)
print('  TEST_DATA  :', TEST_DATA.shape)
print('  TEST_LABEL :', TEST_LABEL.shape)

Raw shapes
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)


In [3]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x'] = out['accel_x'].clip(-128, 127)
    out['accel_y'] = out['accel_y'].clip(-128, 127)
    out['accel_z'] = out['accel_z'].clip(-128, 127)
    out['eda']         = out['eda'].clip(0, 60)
    out['heart_rate']  = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)

def clean_label(df):
    out = df.copy()
    out['id']        = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid']       = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress']    = pd.to_numeric(out['stress'], errors='coerce')
    return out

TRAIN_DATA  = clean_sensor(TRAIN_DATA)
TEST_DATA   = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL  = clean_label(TEST_LABEL)
print('Cleaned.')

Cleaned.


In [4]:
def compute_resting_baselines(sensor_df, low_pct=10):
    refs = {}
    for pid, grp in sensor_df.groupby('pid'):
        hr   = grp['heart_rate'].values.astype(float)
        eda  = grp['eda'].values.astype(float)
        temp = grp['temperature'].values.astype(float)
        valid = np.isfinite(hr) & np.isfinite(eda)
        if valid.sum() < 100:
            refs[pid] = {'hr': float(np.nanmedian(hr)) if valid.any() else 70.0,
                         'eda': float(np.nanmedian(eda)) if valid.any() else 1.0,
                         'temp': float(np.nanmedian(temp)) if valid.any() else 33.0,
                         'hr_std': 5.0, 'eda_std': 0.5,
                         'hr_median': float(np.nanmedian(hr)) if valid.any() else 70.0}
            continue
        hr_v, eda_v, temp_v = hr[valid], eda[valid], temp[valid]
        hr_z  = (hr_v  - hr_v.mean())  / (hr_v.std()  + 1e-9)
        eda_z = (eda_v - eda_v.mean()) / (eda_v.std() + 1e-9)
        arousal  = hr_z + eda_z
        thr      = np.percentile(arousal, low_pct)
        rest_mask = arousal < thr
        if rest_mask.sum() < 10:
            rest_mask = np.ones(len(arousal), dtype=bool)
        refs[pid] = {
            'hr':        float(np.median(hr_v[rest_mask])),
            'eda':       float(np.median(eda_v[rest_mask])),
            'temp':      float(np.median(temp_v[rest_mask])),
            'hr_std':    float(np.std(hr_v[rest_mask]) + 1e-3),
            'eda_std':   float(np.std(eda_v[rest_mask]) + 1e-3),
            'hr_median': float(np.median(hr_v)),
        }
    return refs

TRAIN_REFS = compute_resting_baselines(TRAIN_DATA)
TEST_REFS  = compute_resting_baselines(TEST_DATA)
print('Baselines computed. Train:', len(TRAIN_REFS), '| Test:', len(TEST_REFS))

Baselines computed. Train: 7 | Test: 8


## NEW: Pre-compute session context

For each label, compute its session (same pid, within 30-min gap). Then for each window,
we can look up all other labels in the same session to compute relative rank features.

In [5]:
def assign_sessions(label_df, gap_ms=30*60*1000):
    """Assign session IDs to each label row. Returns label_df with 'session_id' column."""
    out = label_df.copy().sort_values(['pid','timestamp']).reset_index(drop=True)
    sess_id = 0
    session_ids = []
    for pid, grp in out.groupby('pid', sort=False):
        ts   = grp['timestamp'].values.astype(float)
        gaps = np.r_[True, np.diff(ts) > gap_ms]
        for is_new in gaps:
            if is_new: sess_id += 1
            session_ids.append(sess_id)
    out['session_id'] = session_ids
    return out

TRAIN_LABEL_SESS = assign_sessions(TRAIN_LABEL)
TEST_LABEL_SESS  = assign_sessions(TEST_LABEL)
print('Train sessions:', TRAIN_LABEL_SESS['session_id'].nunique())
print('Test  sessions:', TEST_LABEL_SESS['session_id'].nunique())

Train sessions: 67
Test  sessions: 106


## Feature Extraction v25 (v23 features + session-relative + trajectory)

In [6]:
WINDOW_MS  = 180_000
HALF_MS    = 90_000
THIRD_MS   = 60_000
SHORT_MS   = 60_000
LONG_MS    = 300_000
XLONG_MS   = 600_000
TRAJ_A_MS  = 30_000   # trajectory: recent 30s
TRAJ_B_MS  = 90_000   # trajectory: 30s-90s before label

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']: f['hrv_'+k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff) else 0.0
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff) > 25))*100 if len(rr_diff) else 0.0
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff) > 50))*100 if len(rr_diff) else 0.0
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn'] / f['hrv_mean_rr'] if f['hrv_mean_rr'] > 1e-6 else 0.0
    return f

def hrv_frequency_domain(bpm_series):
    f = {'hrv_vlf':np.nan,'hrv_lf':np.nan,'hrv_hf':np.nan,'hrv_lf_hf':np.nan,'hrv_total_power':np.nan}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 60: return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    if len(bpm_1hz) < 30: return f
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_c = rr - rr.mean()
    nperseg = min(len(rr_c), 64)
    if nperseg < 16: return f
    try:
        freqs, psd = sps.welch(rr_c, fs=1.0, nperseg=nperseg, noverlap=nperseg//2, scaling='density')
        def bp(lo, hi):
            mask = (freqs >= lo) & (freqs < hi)
            return float(trapezoid(psd[mask], freqs[mask])) if mask.sum() >= 2 else 0.0
        f['hrv_vlf'] = bp(0.0033,0.04); f['hrv_lf'] = bp(0.04,0.15); f['hrv_hf'] = bp(0.15,0.40)
        f['hrv_total_power'] = f['hrv_vlf'] + f['hrv_lf'] + f['hrv_hf']
        f['hrv_lf_hf'] = f['hrv_lf'] / (f['hrv_hf'] + 1e-6)
    except: pass
    return f

def eda_peak_features(eda_series):
    f = {'eda_n_peaks':np.nan,'eda_peaks_per_min':np.nan,'eda_mean_prominence':np.nan,
         'eda_max_prominence':np.nan,'eda_mean_width':np.nan}
    eda = eda_series.dropna().values.astype(float)
    if len(eda) < 40: return f
    eda_4hz = eda[::8] if len(eda) >= 100 else eda
    if len(eda_4hz) < 16: return f
    try:
        wl = min(len(eda_4hz)-(1 if len(eda_4hz)%2==0 else 0), 15)
        if wl < 5: wl = 5
        if wl % 2 == 0: wl -= 1
        trend = sps.savgol_filter(eda_4hz, window_length=wl, polyorder=2) if len(eda_4hz)>20 else eda_4hz
        phasic = eda_4hz - trend + np.mean(eda_4hz)
        peaks, props = sps.find_peaks(phasic, prominence=0.02, distance=4, width=1)
        f['eda_n_peaks'] = float(len(peaks))
        dur_min = len(eda_4hz)/(4.0*60)
        f['eda_peaks_per_min'] = float(len(peaks)/dur_min) if dur_min > 0 else 0.0
        if len(peaks) > 0:
            f['eda_mean_prominence'] = float(np.mean(props['prominences']))
            f['eda_max_prominence']  = float(np.max(props['prominences']))
            f['eda_mean_width']      = float(np.mean(props['widths']))
        else:
            f['eda_mean_prominence'] = f['eda_max_prominence'] = f['eda_mean_width'] = 0.0
    except: pass
    return f

def eda_tonic_phasic_features(eda_series):
    f = {'eda_tonic_mean':np.nan,'eda_tonic_std':np.nan,'eda_phasic_mean':np.nan,
         'eda_phasic_std':np.nan,'eda_phasic_energy':np.nan,'eda_phasic_max':np.nan}
    eda = eda_series.dropna().values.astype(float)
    if len(eda) < 40: return f
    eda_4hz = eda[::8] if len(eda) >= 100 else eda
    if len(eda_4hz) < 20: return f
    try:
        wlen = min(len(eda_4hz)-(1 if len(eda_4hz)%2==0 else 0), 61)
        if wlen < 5: wlen = 5
        if wlen % 2 == 0: wlen -= 1
        tonic  = sps.savgol_filter(eda_4hz, window_length=wlen, polyorder=1)
        phasic = eda_4hz - tonic
        phasic_pos = np.maximum(phasic, 0)
        f['eda_tonic_mean']    = float(np.mean(tonic))
        f['eda_tonic_std']     = float(np.std(tonic))
        f['eda_phasic_mean']   = float(np.mean(phasic_pos))
        f['eda_phasic_std']    = float(np.std(phasic_pos))
        f['eda_phasic_energy'] = float(np.mean(phasic_pos**2))
        f['eda_phasic_max']    = float(np.max(phasic_pos))
    except: pass
    return f

def accel_jerk_features(ax, ay, az):
    f = {'accel_jerk_mean':np.nan,'accel_jerk_std':np.nan,'accel_jerk_max':np.nan}
    if len(ax) < 5: return f
    try:
        mag  = np.sqrt(ax.astype(float)**2+ay.astype(float)**2+az.astype(float)**2)
        mag  = mag[np.isfinite(mag)]
        if len(mag) < 5: return f
        jerk = np.abs(np.diff(mag))
        f['accel_jerk_mean'] = float(np.mean(jerk))
        f['accel_jerk_std']  = float(np.std(jerk))
        f['accel_jerk_max']  = float(np.max(jerk))
    except: pass
    return f

def timestamp_features(ts_ms):
    try:
        hour = (ts_ms / 3_600_000) % 24.0
        return {'hour_sin':float(np.sin(2*np.pi*hour/24)),
                'hour_cos':float(np.cos(2*np.pi*hour/24)),
                'hour_raw':float(hour)}
    except:
        return {'hour_sin':np.nan,'hour_cos':np.nan,'hour_raw':np.nan}

def extract_features(label_df_sess, sensor_df, pid_enc_map, refs):
    """
    label_df_sess must have 'session_id' column (from assign_sessions).
    """
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    label_ts_by_pid = {pid: np.sort(grp['timestamp'].values.astype(float))
                       for pid, grp in label_df_sess.groupby('pid')}

    # Pre-compute per-session window-mean HR and EDA for rank features
    # We'll build a dict: session_id → list of (timestamp, hr_mean, eda_mean, compound)
    # populated as we go, then used for relative rank
    # Since we process sequentially, we use a two-pass approach:
    # Pass 1: collect all window means. Pass 2: compute ranks.
    # To keep it single-pass-compatible, we collect all first.

    rows_phase1 = []  # (lid, pid, ts, session_id)
    for lrow in label_df_sess.itertuples(index=False):
        rows_phase1.append((int(lrow.id), str(lrow.pid), float(lrow.timestamp), int(lrow.session_id)))

    # Cache window means for all labels first
    win_cache = {}  # lid → {'hr_mean', 'eda_mean', 'compound'}
    for lid, pid, ts, sid in rows_phase1:
        sg = sensor_by_pid.get(pid)
        if sg is None:
            win_cache[lid] = {'whr': np.nan, 'weda': np.nan, 'wcomp': np.nan}
            continue
        ta   = sg['timestamp'].values
        mask = (ta >= ts - WINDOW_MS) & (ta <= ts)
        hr_v = sg.loc[mask, 'heart_rate'].dropna().values.astype(float)
        eda_v= sg.loc[mask, 'eda'].dropna().values.astype(float)
        ref  = refs.get(pid, {})
        hr_m  = float(np.mean(hr_v))  if len(hr_v)  else np.nan
        eda_m = float(np.mean(eda_v)) if len(eda_v) else np.nan
        if ref and np.isfinite(hr_m) and np.isfinite(eda_m):
            hr_z  = (hr_m  - ref['hr'])  / ref['hr_std']
            eda_z = (eda_m - ref['eda']) / ref['eda_std']
            comp  = hr_z + eda_z
        else:
            comp = np.nan
        win_cache[lid] = {'whr': hr_m, 'weda': eda_m, 'wcomp': comp}

    # Build session_id → sorted list of (timestamp, lid) for rank lookup
    sess_to_lids = {}
    for lid, pid, ts, sid in rows_phase1:
        sess_to_lids.setdefault(sid, []).append((ts, lid))
    for sid in sess_to_lids:
        sess_to_lids[sid].sort()

    rows = []
    for n, (lid, pid, ts, sid) in enumerate(rows_phase1, 1):
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat); continue
        ta = sg['timestamp'].values

        # Window slices
        wa     = sg.loc[(ta >= ts - WINDOW_MS) & (ta <= ts), SENSOR_COLS]
        wf     = sg.loc[(ta >= ts - WINDOW_MS) & (ta <  ts - HALF_MS),    SENSOR_COLS]
        wl     = sg.loc[(ta >= ts - HALF_MS)   & (ta <= ts),              SENSOR_COLS]
        wt1    = sg.loc[(ta >= ts - WINDOW_MS) & (ta <  ts - 2*THIRD_MS), SENSOR_COLS]
        wt3    = sg.loc[(ta >= ts - THIRD_MS)  & (ta <= ts),              SENSOR_COLS]
        wshort = sg.loc[(ta >= ts - SHORT_MS)  & (ta <= ts),              SENSOR_COLS]
        wlong  = sg.loc[(ta >= ts - LONG_MS)   & (ta <= ts),              SENSOR_COLS]
        wxlong = sg.loc[(ta >= ts - XLONG_MS)  & (ta <= ts),              SENSOR_COLS]
        # Trajectory windows (NEW)
        w_traj_a = sg.loc[(ta >= ts - TRAJ_A_MS) & (ta <= ts), SENSOR_COLS]             # last 30s
        w_traj_b = sg.loc[(ta >= ts - TRAJ_B_MS) & (ta < ts - TRAJ_A_MS), SENSOR_COLS] # 30-90s ago

        feat['window_count'] = len(wa)
        feat['window_completeness'] = min(1.0, len(wa) / max(WINDOW_MS/1000, 1))

        # Standard v23 features
        for c in SENSOR_COLS:
            v   = wa[c].dropna().values.astype(float)
            vf  = wf[c].dropna().values.astype(float)
            vl  = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean']    = float(np.mean(v))
            feat[f'{c}_std']     = float(np.std(v))
            feat[f'{c}_min']     = float(np.min(v))
            feat[f'{c}_max']     = float(np.max(v))
            feat[f'{c}_median']  = float(np.median(v))
            feat[f'{c}_skew']    = float(spstats.skew(v)) if len(v)>2 else 0.0
            feat[f'{c}_kurt']    = float(spstats.kurtosis(v)) if len(v)>2 else 0.0
            feat[f'{c}_range']   = float(np.max(v)-np.min(v))
            feat[f'{c}_q25']     = float(np.percentile(v,25))
            feat[f'{c}_q75']     = float(np.percentile(v,75))
            feat[f'{c}_iqr']     = feat[f'{c}_q75'] - feat[f'{c}_q25']
            feat[f'{c}_delta']   = float(np.mean(vl)-np.mean(vf)) if len(vf) and len(vl) else 0.0
            feat[f'{c}_slope']   = float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.0
            feat[f'{c}_t1_mean'] = float(np.mean(vt1)) if len(vt1) else float(np.mean(v))
            feat[f'{c}_t3_mean'] = float(np.mean(vt3)) if len(vt3) else float(np.mean(v))
            feat[f'{c}_t3t1']    = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']

        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan
        feat.update(accel_jerk_features(ax, ay, az))
        feat.update(hrv_time_domain(wa['heart_rate']))
        feat.update(hrv_frequency_domain(wa['heart_rate']))
        feat.update(eda_peak_features(wa['eda']))
        feat.update(eda_tonic_phasic_features(wa['eda']))

        for c in ['heart_rate','eda']:
            vs = wshort[c].dropna().values.astype(float)
            if len(vs) == 0:
                feat[f'{c}_short_mean'] = feat[f'{c}_short_std'] = \
                feat[f'{c}_short_max']  = feat[f'{c}_short_slope'] = np.nan
                continue
            feat[f'{c}_short_mean']  = float(np.mean(vs))
            feat[f'{c}_short_std']   = float(np.std(vs))
            feat[f'{c}_short_max']   = float(np.max(vs))
            feat[f'{c}_short_slope'] = float(np.polyfit(np.linspace(0,1,len(vs)),vs,1)[0]) if len(vs)>2 else 0.0

        for c in ['heart_rate','eda','temperature']:
            vl2 = wlong[c].dropna().values.astype(float)
            if len(vl2) == 0:
                feat[f'{c}_long_mean'] = feat[f'{c}_long_std'] = feat[f'{c}_long_slope'] = np.nan
                continue
            feat[f'{c}_long_mean']  = float(np.mean(vl2))
            feat[f'{c}_long_std']   = float(np.std(vl2))
            feat[f'{c}_long_slope'] = float(np.polyfit(np.linspace(0,1,len(vl2)),vl2,1)[0]) if len(vl2)>2 else 0.0

        for c in ['heart_rate','eda']:
            v3 = wa[c].dropna().values.astype(float)
            v5 = wlong[c].dropna().values.astype(float)
            feat[f'{c}_3vs5min'] = float(np.mean(v3)-np.mean(v5)) if len(v3)>0 and len(v5)>0 else np.nan

        for c in ['temperature','heart_rate']:
            vxl = wxlong[c].dropna().values.astype(float)
            if len(vxl) > 2:
                feat[f'{c}_xlong_slope'] = float(np.polyfit(np.linspace(0,1,len(vxl)),vxl,1)[0])
                feat[f'{c}_xlong_mean']  = float(np.mean(vxl))
            else:
                feat[f'{c}_xlong_slope'] = feat[f'{c}_xlong_mean'] = np.nan

        # === NEW: Trajectory features ===
        # Compare last 30s vs 30-90s ago → rising/falling signal
        for c in ['heart_rate', 'eda']:
            va = w_traj_a[c].dropna().values.astype(float)
            vb = w_traj_b[c].dropna().values.astype(float)
            if len(va) > 0 and len(vb) > 0:
                traj_delta = float(np.mean(va) - np.mean(vb))
                feat[f'{c}_traj_delta']    = traj_delta
                feat[f'{c}_traj_rising']   = float(traj_delta > 0)
                feat[f'{c}_traj_fast_rise']= float(traj_delta > np.std(vb) * 0.5) if np.std(vb) > 0 else 0.0
            else:
                feat[f'{c}_traj_delta'] = feat[f'{c}_traj_rising'] = feat[f'{c}_traj_fast_rise'] = np.nan

        # Baseline deviation
        ref = refs.get(pid, {})
        if ref:
            hr_m  = feat.get('heart_rate_mean', np.nan)
            eda_m = feat.get('eda_mean', np.nan)
            tmp_m = feat.get('temperature_mean', np.nan)
            feat['hr_dev_rest']      = (hr_m  - ref['hr'])  if np.isfinite(hr_m)  else np.nan
            feat['hr_dev_rest_std']  = (hr_m  - ref['hr'])  / ref['hr_std']  if np.isfinite(hr_m)  else np.nan
            feat['eda_dev_rest']     = (eda_m - ref['eda']) if np.isfinite(eda_m) else np.nan
            feat['eda_dev_rest_std'] = (eda_m - ref['eda']) / ref['eda_std'] if np.isfinite(eda_m) else np.nan
            feat['temp_dev_rest']    = (tmp_m - ref['temp']) if np.isfinite(tmp_m) else np.nan
            feat['compound_stress']  = feat['hr_dev_rest_std'] + feat['eda_dev_rest_std'] \
                                       if np.isfinite(feat['hr_dev_rest_std']) and np.isfinite(feat['eda_dev_rest_std']) else np.nan
            hr_med = ref.get('hr_median', ref['hr'])
            feat['hr_above_median'] = float(hr_m > hr_med) if np.isfinite(hr_m) else np.nan
            feat['hr_dev_median']   = (hr_m - hr_med)      if np.isfinite(hr_m) else np.nan
        else:
            for k in ['hr_dev_rest','hr_dev_rest_std','eda_dev_rest','eda_dev_rest_std',
                      'temp_dev_rest','compound_stress','hr_above_median','hr_dev_median']:
                feat[k] = np.nan

        # === NEW: Session-relative rank features ===
        # Get all window means in this session, compute rank of current label
        sess_lids = sess_to_lids.get(sid, [])
        if len(sess_lids) >= 2:
            sess_hr    = np.array([win_cache[l]['whr']   for _, l in sess_lids])
            sess_eda   = np.array([win_cache[l]['weda']  for _, l in sess_lids])
            sess_comp  = np.array([win_cache[l]['wcomp'] for _, l in sess_lids])
            cur_hr   = win_cache[lid]['whr']
            cur_eda  = win_cache[lid]['weda']
            cur_comp = win_cache[lid]['wcomp']
            # Percentile rank within session (0=lowest, 1=highest)
            def pct_rank(arr, val):
                valid = arr[np.isfinite(arr)]
                if len(valid) == 0 or not np.isfinite(val): return np.nan
                return float(np.mean(valid <= val))
            feat['sess_hr_rank']   = pct_rank(sess_hr,   cur_hr)
            feat['sess_eda_rank']  = pct_rank(sess_eda,  cur_eda)
            feat['sess_comp_rank'] = pct_rank(sess_comp, cur_comp)
            # Session statistics (context for absolute values)
            feat['sess_hr_mean']   = float(np.nanmean(sess_hr))
            feat['sess_hr_std']    = float(np.nanstd(sess_hr))
            feat['sess_eda_mean']  = float(np.nanmean(sess_eda))
            feat['sess_eda_std']   = float(np.nanstd(sess_eda))
            feat['sess_comp_mean'] = float(np.nanmean(sess_comp))
            feat['sess_comp_std']  = float(np.nanstd(sess_comp))
            # Z-score within session
            feat['sess_hr_zscore']  = (cur_hr  - feat['sess_hr_mean'])  / (feat['sess_hr_std']  + 1e-6) if np.isfinite(cur_hr)  else np.nan
            feat['sess_eda_zscore'] = (cur_eda - feat['sess_eda_mean']) / (feat['sess_eda_std'] + 1e-6) if np.isfinite(cur_eda) else np.nan
            feat['sess_len'] = float(len(sess_lids))
            # Position in session (temporal rank 0=first, 1=last)
            ts_vals = np.array([t for t, _ in sess_lids])
            feat['sess_temporal_rank'] = float(np.mean(ts_vals <= ts))
        else:
            for k in ['sess_hr_rank','sess_eda_rank','sess_comp_rank',
                      'sess_hr_mean','sess_hr_std','sess_eda_mean','sess_eda_std',
                      'sess_comp_mean','sess_comp_std','sess_hr_zscore','sess_eda_zscore',
                      'sess_len','sess_temporal_rank']:
                feat[k] = np.nan

        # Cross-channel correlations
        try:
            hr_a  = wa['heart_rate'].dropna().values.astype(float)
            eda_a = wa['eda'].dropna().values.astype(float)
            tmp_a = wa['temperature'].dropna().values.astype(float)
            nm    = min(len(hr_a), len(eda_a), len(tmp_a))
            if nm >= 30:
                hr_a, eda_a, tmp_a = hr_a[:nm], eda_a[:nm], tmp_a[:nm]
                feat['corr_hr_eda']  = float(np.corrcoef(hr_a,eda_a)[0,1])  if hr_a.std()>1e-6 and eda_a.std()>1e-6  else 0.0
                feat['corr_hr_temp'] = float(np.corrcoef(hr_a,tmp_a)[0,1])  if hr_a.std()>1e-6 and tmp_a.std()>1e-6  else 0.0
                feat['corr_eda_temp']= float(np.corrcoef(eda_a,tmp_a)[0,1]) if eda_a.std()>1e-6 and tmp_a.std()>1e-6 else 0.0
            else:
                feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan
        except:
            feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan

        # Label gap
        try:
            all_ts   = label_ts_by_pid.get(pid, np.array([ts]))
            prior_ts = all_ts[all_ts < ts]
            next_ts  = all_ts[all_ts > ts]
            feat['label_gap_prev_sec'] = float((ts-prior_ts[-1])/1000) if len(prior_ts)>0 else np.nan
            feat['label_gap_next_sec'] = float((next_ts[0]-ts)/1000)   if len(next_ts)>0  else np.nan
        except:
            feat['label_gap_prev_sec'] = feat['label_gap_next_sec'] = np.nan

        feat.update(timestamp_features(ts))
        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(rows_phase1)}')
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p: i for i, p in enumerate(TRAIN_LABEL['pid'].unique())}
print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL_SESS, TRAIN_DATA, train_pid_map, TRAIN_REFS)
print('Extracting test features...')
test_features  = extract_features(TEST_LABEL_SESS,  TEST_DATA,  train_pid_map, TEST_REFS)
print(f'train: {train_features.shape} | test: {test_features.shape}')

Extracting train features...
  200/815
  400/815
  600/815
  800/815
Extracting test features...
  200/1028
  400/1028
  600/1028
  800/1028
  1000/1028
train: (815, 185) | test: (1028, 185)


In [7]:
tli = TRAIN_LABEL.set_index('id')
y   = tli.loc[train_features.index, 'stress'].astype(int)

common_cols    = [c for c in train_features.columns if c in test_features.columns]
train_features = train_features[common_cols]
test_features  = test_features[common_cols]

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),  columns=common_cols, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),       columns=common_cols, index=test_features.index)

counts = Counter(y); total = len(y)
print('Class dist:', dict(counts))
# Uncapped weights
class_weights  = {k: total / (3 * counts[k]) for k in [0,1,2]}
sample_weights = np.array([class_weights[int(yi)] for yi in y])
train_prior    = np.array([counts[i]/total for i in range(3)])
print('Class weights:', {k: round(v,3) for k,v in class_weights.items()})
print('Train prior:', train_prior.round(3).tolist())
print(f'New session-relative features count: {sum(1 for c in common_cols if "sess_" in c or "traj_" in c)}')

Class dist: {2: 587, 0: 162, 1: 66}
Class weights: {0: 1.677, 1: 4.116, 2: 0.463}
Train prior: [0.199, 0.081, 0.72]
New session-relative features count: 19


## SMOTE (class 1)

In [8]:
def smote_minority(X, y, minority_class=1, n_copies=3, noise_std_frac=0.01, seed=42):
    rng    = np.random.RandomState(seed)
    mask   = (y == minority_class).values
    X_min  = X.values[mask]
    y_min  = y.values[mask]
    col_stds = X.values.std(axis=0) * noise_std_frac
    X_aug_parts = [X.values]
    y_aug_parts = [y.values]
    for _ in range(n_copies):
        noise = rng.randn(*X_min.shape) * col_stds
        X_aug_parts.append(X_min + noise)
        y_aug_parts.append(y_min)
    X_aug = np.vstack(X_aug_parts)
    y_aug = np.concatenate(y_aug_parts)
    idx   = rng.permutation(len(y_aug))
    return pd.DataFrame(X_aug[idx], columns=X.columns), pd.Series(y_aug[idx], name=y.name)

X_aug, y_aug = smote_minority(X_imp, y, minority_class=1, n_copies=3, noise_std_frac=0.01)
counts_aug     = Counter(y_aug)
total_aug      = len(y_aug)
sw_aug         = np.array([total_aug/(3*counts_aug[int(yi)]) for yi in y_aug])
print(f'After SMOTE — dist: {dict(counts_aug)} | total rows: {total_aug}')

After SMOTE — dist: {0: 162, 2: 587, 1: 264} | total rows: 1013


## Sessions

In [9]:
def make_session_groups_from_sess(label_df_sess):
    """Use already-assigned session_ids."""
    label_df_sess = label_df_sess.copy().reset_index(drop=True)
    out = []
    for sid, grp in label_df_sess.groupby('session_id'):
        out.append(grp.index.values)
    return out

def smooth_by_session(proba, sessions, strength=0.20):
    out = proba.copy()
    for idx in sessions:
        mean = proba[idx].mean(axis=0, keepdims=True)
        out[idx] = (1 - strength) * proba[idx] + strength * mean
    return out

train_label_aligned = TRAIN_LABEL_SESS.set_index('id').loc[X_imp.index].reset_index()
TRAIN_SESSIONS = make_session_groups_from_sess(train_label_aligned)
TEST_SESSIONS  = make_session_groups_from_sess(TEST_LABEL_SESS.reset_index(drop=True))
print('Train sessions:', len(TRAIN_SESSIONS), '| Test sessions:', len(TEST_SESSIONS))

Train sessions: 67 | Test sessions: 106


## Multi-model ensemble: LightGBM + CatBoost + ExtraTrees

Phase 1: OOF probabilities on original data (for isotonic calibration).
Phase 2: Retrain on augmented data for test predictions.

In [10]:
LGBM_PARAMS = dict(
    n_estimators=1500, learning_rate=0.02, num_leaves=63, max_depth=-1,
    min_child_samples=20, subsample=0.6, subsample_freq=1, colsample_bytree=0.4,
    reg_alpha=0.3, reg_lambda=0.5, class_weight='balanced',
    objective='multiclass', num_class=3, n_jobs=-1, verbose=-1,
)

CAT_PARAMS = dict(
    iterations=800, learning_rate=0.03, depth=6,
    l2_leaf_reg=5.0, min_data_in_leaf=15,
    loss_function='MultiClass', eval_metric='TotalF1',
    class_weights=[class_weights[0], class_weights[1], class_weights[2]],
    random_seed=42, verbose=0,
)

ET_PARAMS = dict(
    n_estimators=500, max_depth=None, min_samples_leaf=5,
    max_features=0.4, class_weight='balanced', n_jobs=-1, random_state=42,
)

SEEDS    = [42, 7, 123, 17, 99, 256, 314, 888, 512, 2024]
N_SPLITS = 5

# Ensemble weights (tuned by typical relative CV performance; LGBM usually best)
LGBM_W = 0.50
CAT_W  = 0.30
ET_W   = 0.20

print('=== Phase 1: OOF (original data, for isotonic calibration) ===')
oof_proba_lgbm = np.zeros((len(X_imp), 3))
oof_proba_cat  = np.zeros((len(X_imp), 3))
oof_proba_et   = np.zeros((len(X_imp), 3))
oof_count      = np.zeros(len(X_imp))
all_cv_scores  = []

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y), 1):
        Xtr, Xval = X_imp.iloc[tr_idx], X_imp.iloc[val_idx]
        ytr, yval = y.iloc[tr_idx],     y.iloc[val_idx]
        swtr       = sample_weights[tr_idx]

        # LightGBM
        lgbm_m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        lgbm_m.fit(Xtr, ytr, sample_weight=swtr,
                   eval_set=[(Xval, yval)],
                   callbacks=[lgb.early_stopping(150,verbose=False), lgb.log_evaluation(-1)])
        p_lgbm = lgbm_m.predict_proba(Xval)

        # CatBoost
        cat_m = CatBoostClassifier(**{**CAT_PARAMS, 'random_seed': seed})
        cat_m.fit(Xtr, ytr, sample_weight=swtr,
                  eval_set=(Xval, yval), use_best_model=True, early_stopping_rounds=100)
        p_cat = cat_m.predict_proba(Xval)

        # ExtraTrees
        et_m = ExtraTreesClassifier(**{**ET_PARAMS, 'random_state': seed})
        et_m.fit(Xtr, ytr, sample_weight=swtr)
        p_et = et_m.predict_proba(Xval)

        p_ensemble = LGBM_W*p_lgbm + CAT_W*p_cat + ET_W*p_et
        score = balanced_accuracy_score(yval, p_ensemble.argmax(1))
        fold_scores.append(score)

        oof_proba_lgbm[val_idx] += p_lgbm
        oof_proba_cat[val_idx]  += p_cat
        oof_proba_et[val_idx]   += p_et
        oof_count[val_idx]      += 1

    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed:4d}: OOF CV BA = {np.mean(fold_scores):.4f}')

oof_proba_lgbm /= oof_count[:, None]
oof_proba_cat  /= oof_count[:, None]
oof_proba_et   /= oof_count[:, None]
oof_ensemble    = LGBM_W*oof_proba_lgbm + CAT_W*oof_proba_cat + ET_W*oof_proba_et

print(f'\nMean OOF ensemble BA: {np.mean(all_cv_scores):.4f}')
print('OOF argmax dist:', dict(Counter(oof_ensemble.argmax(1))))

=== Phase 1: OOF (original data, for isotonic calibration) ===
  Seed   42: OOF CV BA = 0.9607
  Seed    7: OOF CV BA = 0.9691
  Seed  123: OOF CV BA = 0.9318
  Seed   17: OOF CV BA = 0.9454
  Seed   99: OOF CV BA = 0.9632
  Seed  256: OOF CV BA = 0.9650
  Seed  314: OOF CV BA = 0.9585
  Seed  888: OOF CV BA = 0.9735
  Seed  512: OOF CV BA = 0.9490
  Seed 2024: OOF CV BA = 0.9702

Mean OOF ensemble BA: 0.9586
OOF argmax dist: {np.int64(2): 565, np.int64(0): 186, np.int64(1): 64}


## Isotonic calibration on OOF

In [11]:
# Fit one isotonic regressor per class on OOF probabilities
# This reshapes probability outputs to be better calibrated
y_bin = label_binarize(y.values, classes=[0,1,2])  # (N, 3)

iso_regs = []
oof_cal   = np.zeros_like(oof_ensemble)
for c in range(3):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(oof_ensemble[:, c], y_bin[:, c])
    oof_cal[:, c] = iso.predict(oof_ensemble[:, c])
    iso_regs.append(iso)

# Renormalize
oof_cal = oof_cal / (oof_cal.sum(axis=1, keepdims=True) + 1e-9)

ba_raw = balanced_accuracy_score(y.values, oof_ensemble.argmax(1))
ba_cal = balanced_accuracy_score(y.values, oof_cal.argmax(1))
print(f'OOF BA before isotonic: {ba_raw:.4f}')
print(f'OOF BA after  isotonic: {ba_cal:.4f}')
print(f'OOF dist raw: {dict(Counter(oof_ensemble.argmax(1)))}')
print(f'OOF dist cal: {dict(Counter(oof_cal.argmax(1)))}')

OOF BA before isotonic: 0.9645
OOF BA after  isotonic: 0.9816
OOF dist raw: {np.int64(2): 565, np.int64(0): 186, np.int64(1): 64}
OOF dist cal: {np.int64(2): 585, np.int64(0): 161, np.int64(1): 69}


In [12]:
print('=== Phase 2: Test inference (augmented data) ===')
test_proba_lgbm = np.zeros((len(X_test_imp), 3))
test_proba_cat  = np.zeros((len(X_test_imp), 3))
test_proba_et   = np.zeros((len(X_test_imp), 3))

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_lgbm = np.zeros((len(X_test_imp), 3))
    seed_cat  = np.zeros((len(X_test_imp), 3))
    seed_et   = np.zeros((len(X_test_imp), 3))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_aug, y_aug), 1):
        Xtr = X_aug.iloc[tr_idx]; ytr = y_aug.iloc[tr_idx]; swtr = sw_aug[tr_idx]
        Xval_aug = X_aug.iloc[val_idx]; yval_aug = y_aug.iloc[val_idx]

        # LightGBM
        lgbm_m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        lgbm_m.fit(Xtr, ytr, sample_weight=swtr,
                   eval_set=[(Xval_aug, yval_aug)],
                   callbacks=[lgb.early_stopping(150,verbose=False), lgb.log_evaluation(-1)])
        seed_lgbm += lgbm_m.predict_proba(X_test_imp)

        # CatBoost
        cat_m = CatBoostClassifier(**{**CAT_PARAMS, 'random_seed': seed})
        cat_m.fit(Xtr, ytr, sample_weight=swtr,
                  eval_set=(Xval_aug, yval_aug), use_best_model=True, early_stopping_rounds=100)
        seed_cat += cat_m.predict_proba(X_test_imp)

        # ExtraTrees
        et_m = ExtraTreesClassifier(**{**ET_PARAMS, 'random_state': seed})
        et_m.fit(Xtr, ytr, sample_weight=swtr)
        seed_et += et_m.predict_proba(X_test_imp)

    test_proba_lgbm += seed_lgbm / N_SPLITS
    test_proba_cat  += seed_cat  / N_SPLITS
    test_proba_et   += seed_et   / N_SPLITS
    print(f'  Seed {seed:4d} done')

test_proba_lgbm /= len(SEEDS)
test_proba_cat  /= len(SEEDS)
test_proba_et   /= len(SEEDS)
raw_test_proba   = LGBM_W*test_proba_lgbm + CAT_W*test_proba_cat + ET_W*test_proba_et

print('\nRaw test argmax dist:', dict(Counter(raw_test_proba.argmax(1))))
print('LGBM test argmax dist:', dict(Counter(test_proba_lgbm.argmax(1))))
print('CAT  test argmax dist:', dict(Counter(test_proba_cat.argmax(1))))
print('ET   test argmax dist:', dict(Counter(test_proba_et.argmax(1))))

=== Phase 2: Test inference (augmented data) ===
  Seed   42 done
  Seed    7 done
  Seed  123 done
  Seed   17 done
  Seed   99 done
  Seed  256 done
  Seed  314 done
  Seed  888 done
  Seed  512 done
  Seed 2024 done

Raw test argmax dist: {np.int64(0): 366, np.int64(2): 551, np.int64(1): 111}
LGBM test argmax dist: {np.int64(0): 266, np.int64(2): 628, np.int64(1): 134}
CAT  test argmax dist: {np.int64(0): 554, np.int64(1): 71, np.int64(2): 403}
ET   test argmax dist: {np.int64(2): 728, np.int64(0): 221, np.int64(1): 79}


## Apply isotonic calibration to test + submission grid

In [13]:
def apply_isotonic(proba, iso_regs):
    cal = np.zeros_like(proba)
    for c in range(3):
        cal[:, c] = iso_regs[c].predict(proba[:, c])
    cal = cal / (cal.sum(axis=1, keepdims=True) + 1e-9)
    return cal

test_proba_cal = apply_isotonic(raw_test_proba, iso_regs)
print('Test after isotonic dist:', dict(Counter(test_proba_cal.argmax(1))))

def make_session_groups_idx(label_df, gap_ms=30*60*1000):
    """Simple index-based session groups for submission ordering."""
    labels = label_df.copy().reset_index(drop=True)
    out = []
    for pid, grp in labels.sort_values(['pid','timestamp']).groupby('pid', sort=False):
        ts   = grp['timestamp'].values.astype(float)
        sess = np.cumsum(np.r_[0, np.diff(ts) > gap_ms])
        for sid in np.unique(sess):
            out.append(grp.index.values[sess == sid])
    return out

TEST_SESS_IDX = make_session_groups_idx(TEST_LABEL)

def make_submission(proba, alpha, smooth_strength, sessions, prior, fname):
    cal = proba * (prior ** alpha)
    cal = cal / cal.sum(axis=1, keepdims=True)
    if smooth_strength > 0:
        cal = smooth_by_session(cal, sessions, strength=smooth_strength)
    preds = cal.argmax(1).astype(int)
    pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds}).to_csv(fname, index=False)
    cnts  = np.bincount(preds, minlength=3)
    dev   = np.abs(cnts/len(preds) - prior).max()
    return cnts, dev

print(f'\n{"file":>55s}  {"source":>6s} {"a":>4s} {"sm":>5s}  {"dist":>22s}  {"dev":>5s}')

all_results = []
best_dev = np.inf; best_fname = None

for source_name, proba in [("raw", raw_test_proba), ("iso", test_proba_cal)]:
    for alpha in [0.8, 1.0, 1.2]:
        for smooth in [0.20, 0.25, 0.30]:
            fname = f'submission_{source_name}_a{alpha}_s{smooth}.csv'
            cnts, dev = make_submission(proba, alpha, smooth, TEST_SESS_IDX, train_prior, fname)
            all_results.append((dev, source_name, alpha, smooth, fname, cnts))
            marker = ' <<<' if dev < best_dev else ''
            if dev < best_dev: best_dev = dev; best_fname = fname
            print(f'{fname:>55s}  {source_name:>6s} {alpha:>4.1f} {smooth:>5.2f}  {str(cnts.tolist()):>22s}  {dev:>5.3f}{marker}')

import shutil
shutil.copy(best_fname, 'submission_best.csv')
print(f'\n>>> Best: {best_fname} (dev={best_dev:.3f})')

Test after isotonic dist: {np.int64(2): 815, np.int64(1): 129, np.int64(0): 84}

                                                   file  source    a    sm                    dist    dev
                           submission_raw_a0.8_s0.2.csv     raw  0.8  0.20           [165, 3, 860]  0.116 <<<
                          submission_raw_a0.8_s0.25.csv     raw  0.8  0.25           [161, 2, 865]  0.121
                           submission_raw_a0.8_s0.3.csv     raw  0.8  0.30           [154, 2, 872]  0.128
                           submission_raw_a1.0_s0.2.csv     raw  1.0  0.20           [129, 1, 898]  0.153
                          submission_raw_a1.0_s0.25.csv     raw  1.0  0.25           [126, 0, 902]  0.157
                           submission_raw_a1.0_s0.3.csv     raw  1.0  0.30           [118, 0, 910]  0.165
                           submission_raw_a1.2_s0.2.csv     raw  1.2  0.20           [100, 0, 928]  0.182
                          submission_raw_a1.2_s0.25.csv     raw  1.

In [14]:
print('========== v25 SUMMARY ==========')
print(f'Features: {X_imp.shape[1]} (incl. {sum(1 for c in X_imp.columns if "sess_" in c or "traj_" in c)} new session/traj)')
print(f'Models: LGBM {LGBM_W:.0%} + CatBoost {CAT_W:.0%} + ExtraTrees {ET_W:.0%}')
print(f'Seeds × folds: {len(SEEDS)} × {N_SPLITS} = {len(SEEDS)*N_SPLITS} models per family')
print(f'Mean OOF ensemble BA: {np.mean(all_cv_scores):.4f}')
print(f'OOF BA after isotonic: {ba_cal:.4f}')
print()
print('Top 5 submissions by dist dev:')
for dev, src, a, sm, fn, cnts in sorted(all_results)[:5]:
    print(f'  {fn}  dist={cnts.tolist()}  dev={dev:.3f}')
print()
print('SUBMISSION ORDER:')
print('  1. submission_best.csv   ← auto-selected, lowest dist dev')
print('  2. Check the table for raw vs iso comparison at alpha=0.8')
print('  Expected: iso (isotonic-calibrated) version should have better-shaped proba')
print('==================================')

========== v25 SUMMARY ==========
Features: 185 (incl. 19 new session/traj)
Models: LGBM 50% + CatBoost 30% + ExtraTrees 20%
Seeds × folds: 10 × 5 = 50 models per family
Mean OOF ensemble BA: 0.9586
OOF BA after isotonic: 0.9816

Top 5 submissions by dist dev:
  submission_raw_a0.8_s0.2.csv  dist=[165, 3, 860]  dev=0.116
  submission_raw_a0.8_s0.25.csv  dist=[161, 2, 865]  dev=0.121
  submission_raw_a0.8_s0.3.csv  dist=[154, 2, 872]  dev=0.128
  submission_iso_a0.8_s0.2.csv  dist=[75, 73, 880]  dev=0.136
  submission_iso_a0.8_s0.25.csv  dist=[75, 73, 880]  dev=0.136

SUBMISSION ORDER:
  1. submission_best.csv   ← auto-selected, lowest dist dev
  2. Check the table for raw vs iso comparison at alpha=0.8
  Expected: iso (isotonic-calibrated) version should have better-shaped proba
